<h1 style="text-align: center">SESAR study</h1>


In [1]:
import pandas as pd
import seaborn as sns
import os
from os.path import basename
from os import getcwd, chdir, mkdir
import subprocess
import matplotlib.pyplot as plt
import numpy as np 
import shutil
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import ipywidgets as widgets
import pickle as pk
from scipy.interpolate import interp1d

/tmp/ipykernel_19179/3663016583.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
# Working directories
path_main = "/projects/fms_exp/uerrxn0d/WORK/SESAR_4D_TBO_v3/"

path_p_simu = path_main + "p_simu"
path_reports = path_main + "reports"

Name_HTML_Report = "Report_name"


FMS_MODEL = "FMS/A330/A330-941-VG10"
AC_name = "A330-941"

# Inputs

CG = 0.35
Rating = "MCL"
VMO = 330.0
MMO = 0.86

GW_values = [170000.0, 200000.0, 220000.0]
DISA_values = [0.0, 15.0, 25.0]
CI_values = [0.0, 50.0, 100.0]

plot_color_ci = {0.0 : "blue", 50.0 : "magenta", 100.0 : "orange"}




In [3]:
%%capture

##################################
# OPTALT computations
##################################
os.chdir(path_p_simu)

GW = "\n".join(str(gw) for gw in GW_values)
DISA = "\n".join(str(disa) for disa in DISA_values)

with open("OPTALT.txt", "w") as file:
    file.write('''@OUTPUT GW ZP_CRUISE  ZMAX "TYP OPT"  CI VENT DISA  XCG  ZOPT FMS_MODEL

############################### INPUT PARAMETERS ########################################

%FMS_MODEL
'''+ str(FMS_MODEL) + '''

%DISA
'''+ str(DISA) + '''

%GW
'''+ str(GW) + '''

%CPSD_USE     CPSD_PATH
False         "not used"

%MOTEUR_EXTERNE_DROIT
On

%MOTEUR_EXTERNE_GAUCHE
On

%ANTI_ICE
On

%VENT_CONSTANT
Oui

%DISA_CONSTANT
Oui

%DCX
0.

%PERFF
0.

%STANDARD_FMS
"FMS TAV post REV2"

%OPTIONS_AVANCEES
Oui

%ABSCISSA_VENT
0.0

%ALT_VENT 
0.00000000E+00 

%VITESSE_VENT
0.00000000E+00

%ABSCISSA_DISA
0.0

%ALT_DISA
0.00000000E+00

%VALUE_DISA
0.00000000E+00

%CRUISE_ALT
3000.

%VENT
0.0

%COST_INDEX
0.

%CG
'''+ str(CG) + '''

''')
    
!ps_auto OPTALT.txt -s OPTALT -o OPTALT.csv

OPTALT_df = pd.read_csv('OPTALT.csv',sep='\t')


In [4]:
#Prepare inputs

data_list = []

for gw in GW_values:
    for disa in DISA_values:

        opt_fl = OPTALT_df[(OPTALT_df["GW"] == gw) & (OPTALT_df["DISA"] == disa)]["ZOPT"].values[0]
        optalt = opt_fl * 100

        data_dict = {                    
                    "DISA": disa,
                    "GW_INIT": gw,
                    "ZP_CRZ": optalt,
                    }

        data_list.append(data_dict)

df_input = pd.DataFrame(data_list)
        
input_data = df_input.to_csv(sep='\t', index=False)


In [5]:
%%capture

#Climb at Greendot speed
os.chdir(path_p_simu)

with open("Climb_GDOT.txt", "w") as file:
    file.write('''#!  -s ./template_climb_GDOT.vol --nosort -v -o ./Res_Climb_GDOT.csv

@OUTPUT INPUT_name  MODELE  GW_INIT
@OUTPUT DIST CONSO T ZP MACH CAS VSOLH GW  FRAM FG
@OUTPUT TAU TSPD WFE  CZ  CX ALPHA GAMA_AIR_GEOM
@OUTPUT VZP VENT DISA XCG DXCG TSPR TAS N1K CONF FEP
@OUTPUT "Type du segment" "Type sous segment"

%  MODELE
'''+ str(FMS_MODEL) + '''

%RATING
'''+ str(Rating) + '''

% CG 
'''+ str(CG) + '''

%'''+ input_data)

!ps_auto Climb_GDOT.txt -s plan de vol

Res_Climb_GDOT_df = pd.read_csv('Res_Climb_GDOT.csv',sep='\t')
Res_Climb_GDOT_df.rename(columns={'T': 'TIME'}, inplace=True)
# display(Res_Climb_df)


In [6]:
%%capture

#Climb at Cost Index
os.chdir(path_p_simu)

CI = "\n".join(str(ci) for ci in CI_values)

with open("Climb_CI.txt", "w") as file:
    file.write('''#!  -s ./template_climb_CI.vol --nosort -v -o ./Res_Climb_CI.csv

@OUTPUT INPUT_name  MODELE  GW_INIT  CI
@OUTPUT DIST CONSO T ZP MACH CAS VSOLH GW  FRAM FG
@OUTPUT TAU TSPD WFE  CZ  CX ALPHA GAMA_AIR_GEOM
@OUTPUT VZP VENT DISA XCG DXCG TSPR TAS N1K CONF FEP
@OUTPUT "Type du segment" "Type sous segment"

%  MODELE
'''+ str(FMS_MODEL) + '''

%RATING
'''+ str(Rating) + '''

% CG 
'''+ str(CG) + '''

% CI 
'''+ str(CI) + '''

%'''+ input_data)

!ps_auto Climb_CI.txt -s plan de vol

Res_Climb_CI_df = pd.read_csv('Res_Climb_CI.csv',sep='\t')
Res_Climb_CI_df.rename(columns={'T': 'TIME'}, inplace=True)
# display(Res_Climb_CI_df)


In [7]:
%%capture

#Climb at CAS MACH
os.chdir(path_p_simu)

with open("Climb_CAS_MACH.txt", "w") as file:
    file.write('''#!  -s ./template_climb_CAS_Mach.vol --nosort -v -o ./Res_Climb_CAS_MACH.csv

@OUTPUT INPUT_name  MODELE  GW_INIT
@OUTPUT DIST CONSO T ZP MACH CAS VSOLH GW  FRAM FG
@OUTPUT TAU TSPD WFE  CZ  CX ALPHA GAMA_AIR_GEOM
@OUTPUT VZP VENT DISA XCG DXCG TSPR TAS N1K CONF FEP
@OUTPUT "Type du segment" "Type sous segment"

%  MODELE
'''+ str(FMS_MODEL) + '''

%RATING
'''+ str(Rating) + '''

% CG 
'''+ str(CG) + '''

% CLIMB_CAS 
'''+ str(VMO) + '''

% CLIMB_MACH 
'''+ str(MMO) + '''

%'''+ input_data)

!ps_auto Climb_CAS_MACH.txt -s plan de vol

Res_Climb_CAS_MACH_df = pd.read_csv('Res_Climb_CAS_MACH.csv',sep='\t')
Res_Climb_CAS_MACH_df.rename(columns={'T': 'TIME'}, inplace=True)
display(Res_Climb_CAS_MACH_df)


## Annexes

In [8]:
os.chdir(path_reports)

data_list_CAS_MACH = []
data_list_GDOT = []
data_list_CI = []

for gw in GW_values:
    Res_Climb_CAS_MACH_df_tmp1 = Res_Climb_CAS_MACH_df[Res_Climb_CAS_MACH_df["GW_INIT"] == gw]
    Res_Climb_CI_df_tmp1 = Res_Climb_CI_df[Res_Climb_CI_df["GW_INIT"] == gw]
    Res_Climb_GDOT_df_tmp1 = Res_Climb_GDOT_df[Res_Climb_GDOT_df["GW_INIT"] == gw] 
    
    gw_t = gw / 1000
    
    for disa in DISA_values:
        Res_Climb_CAS_MACH_df_tmp2 = Res_Climb_CAS_MACH_df_tmp1[Res_Climb_CAS_MACH_df_tmp1["DISA"] == disa]
        Res_Climb_CI_df_tmp2 = Res_Climb_CI_df_tmp1[Res_Climb_CI_df_tmp1["DISA"] == disa]
        Res_Climb_GDOT_df_tmp2 = Res_Climb_GDOT_df_tmp1[Res_Climb_GDOT_df_tmp1["DISA"] == disa]
        
        zp_crz = Res_Climb_GDOT_df_tmp2["ZP"].max()
        ZP_values_tmp = [10000.0, 12000.0, 20000.0, 30000.0]
        
        ZP_values = [x for x in ZP_values_tmp if x < zp_crz]
        ZP_values.append(zp_crz)
        
        Res_Climb_CI0 = Res_Climb_CI_df_tmp2[Res_Climb_CI_df_tmp2["CI"] == 0]        
        
        for zp in ZP_values:
            
            f_interp_Time_CI0 = interp1d(Res_Climb_CI0["ZP"], Res_Climb_CI0["TIME"], fill_value='extrapolate')
            time_CI0 = f_interp_Time_CI0(zp)
            
            f_interp_Dist_CI0 = interp1d(Res_Climb_CI0["ZP"], Res_Climb_CI0["DIST"], fill_value='extrapolate')
            dist_CI0 = f_interp_Dist_CI0(zp)
            
            #Results VMO-MMO
            f_interp_Time_Climb_CAS_MACH = interp1d(Res_Climb_CAS_MACH_df_tmp2["ZP"], Res_Climb_CAS_MACH_df_tmp2["TIME"], fill_value='extrapolate')
            time_mmo_vmo = f_interp_Time_Climb_CAS_MACH(zp)
            
            f_interp_Dist_Climb_CAS_MACH = interp1d(Res_Climb_CAS_MACH_df_tmp2["ZP"], Res_Climb_CAS_MACH_df_tmp2["DIST"], fill_value='extrapolate')
            dist_mmo_vmo = f_interp_Dist_Climb_CAS_MACH(zp)
            
            delta_time_mmo_vmo = time_mmo_vmo - time_CI0
            delta_dist_mmo_vmo = dist_mmo_vmo - dist_CI0
            
            data_dict_CAS_MACH = {
                "GW": gw,
                "DISA": disa,
                "ZP": zp,
                "TIME": time_mmo_vmo,
                "DIST": dist_mmo_vmo,        
                "DELTA TIME vs CI0": delta_time_mmo_vmo,
                "DELTA DIST vs CI0": delta_dist_mmo_vmo,
            }

            data_list_CAS_MACH.append(data_dict_CAS_MACH)
            
            #Results GDOT
            f_interp_Time_Climb_GDOT = interp1d(Res_Climb_GDOT_df_tmp2["ZP"], Res_Climb_GDOT_df_tmp2["TIME"], fill_value='extrapolate')
            time_gdot = f_interp_Time_Climb_GDOT(zp)
            
            f_interp_Dist_Climb_GDOT = interp1d(Res_Climb_GDOT_df_tmp2["ZP"], Res_Climb_GDOT_df_tmp2["DIST"], fill_value='extrapolate')
            dist_gdot = f_interp_Dist_Climb_GDOT(zp)
            
            delta_time_gdot = time_gdot - time_CI0
            delta_dist_gdot = dist_gdot - dist_CI0
            
            data_dict_GDOT = {
                "GW": gw,
                "DISA": disa,
                "ZP": zp,
                "TIME": time_gdot,
                "DIST": dist_gdot,        
                "DELTA TIME vs CI0": delta_time_gdot,
                "DELTA DIST vs CI0": delta_dist_gdot,
            }

            data_list_GDOT.append(data_dict_GDOT)
            
            #Results Cost Index
            for ci in CI_values:            
                Res_Climb_CI_df_tmp3 = Res_Climb_CI_df_tmp2[Res_Climb_CI_df_tmp2["CI"] == ci]
                
                f_interp_Time_Climb_CI = interp1d(Res_Climb_CI_df_tmp3["ZP"], Res_Climb_CI_df_tmp3["TIME"], fill_value='extrapolate')
                time_CI = f_interp_Time_Climb_CI(zp)

                f_interp_Dist_Climb_CI = interp1d(Res_Climb_CI_df_tmp3["ZP"], Res_Climb_CI_df_tmp3["DIST"], fill_value='extrapolate')
                dist_CI = f_interp_Dist_Climb_CI(zp)

                delta_time_CI = time_CI - time_CI0
                delta_dist_CI = dist_CI - dist_CI0

                data_dict_CI = {
                    "GW": gw,
                    "DISA": disa,
                    "ZP": zp,
                    "CI": ci,
                    "TIME": time_CI,
                    "DIST": dist_CI,        
                    "DELTA TIME vs CI0": delta_time_CI,
                    "DELTA DIST vs CI0": delta_dist_CI,
                }

                data_list_CI.append(data_dict_CI)
            
df_res_CAS_MACH = pd.DataFrame(data_list_CAS_MACH)
df_res_CAS_MACH.to_csv(f"{AC_name}_VMO_MMO_res.csv", index=False)
# display(df_res_CAS_MACH)
            
df_res_GDOT = pd.DataFrame(data_list_GDOT)
df_res_GDOT.to_csv(f"{AC_name}_GDOT_res.csv", index=False)
# display(df_res_GDOT)

df_res_CI = pd.DataFrame(data_list_CI)
df_res_CI.to_csv(f"{AC_name}_CI_res.csv", index=False)
# display(df_res_CI)

In [1]:
plt.style.use('seaborn-whitegrid')

for gw in GW_values:
    Res_Climb_CAS_MACH_df_tmp1 = Res_Climb_CAS_MACH_df[Res_Climb_CAS_MACH_df["GW_INIT"] == gw]
    Res_Climb_CI_df_tmp1 = Res_Climb_CI_df[Res_Climb_CI_df["GW_INIT"] == gw]
    Res_Climb_GDOT_df_tmp1 = Res_Climb_GDOT_df[Res_Climb_GDOT_df["GW_INIT"] == gw] 
    
    gw_t = gw / 1000
    
    for disa in DISA_values:
        Res_Climb_CAS_MACH_df_tmp2 = Res_Climb_CAS_MACH_df_tmp1[Res_Climb_CAS_MACH_df_tmp1["DISA"] == disa]
        Res_Climb_CI_df_tmp2 = Res_Climb_CI_df_tmp1[Res_Climb_CI_df_tmp1["DISA"] == disa]
        Res_Climb_GDOT_df_tmp2 = Res_Climb_GDOT_df_tmp1[Res_Climb_GDOT_df_tmp1["DISA"] == disa] 
#         display(Res_Climb_CAS_MACH_df_tmp2)
            
        fig, ax = plt.subplots(nrows=2, ncols=3,figsize=(30, 12), squeeze=False)
        
        for ci in CI_values:
            
            Res_Climb_CI_df_tmp3 = Res_Climb_CI_df_tmp2[Res_Climb_CI_df_tmp2["CI"] == ci]
            
            plot_color = plot_color_ci[ci]
            
            ax[0,0].plot(Res_Climb_CI_df_tmp3.DIST, Res_Climb_CI_df_tmp3.ZP, label = f"CI{ci}", linestyle = 'solid', color = plot_color)
            ax[0,1].plot(Res_Climb_CI_df_tmp3.DIST, Res_Climb_CI_df_tmp3.CAS, label = f"CI{ci}", linestyle = 'solid', color = plot_color)
            ax[0,2].plot(Res_Climb_CI_df_tmp3.DIST, Res_Climb_CI_df_tmp3.MACH, label = f"CI{ci}", linestyle = 'solid', color = plot_color)
            ax[1,0].plot(Res_Climb_CI_df_tmp3.TIME, Res_Climb_CI_df_tmp3.ZP, label = f"CI{ci}", linestyle = 'solid', color = plot_color) 
            ax[1,1].plot(Res_Climb_CI_df_tmp3.ZP, Res_Climb_CI_df_tmp3.VZP, label = f"CI{ci}", linestyle = 'solid', color = plot_color) 
            ax[1,2].plot(Res_Climb_CI_df_tmp3.ZP, Res_Climb_CI_df_tmp3.GAMA_AIR_GEOM, label = f"CI{ci}", linestyle = 'solid', color = plot_color) 

        ax[0,0].plot(Res_Climb_CAS_MACH_df_tmp2.DIST, Res_Climb_CAS_MACH_df_tmp2.ZP, label = "VMO / MMO", linestyle = 'solid', color = "red")
        ax[0,0].plot(Res_Climb_GDOT_df_tmp2.DIST, Res_Climb_GDOT_df_tmp2.ZP, label = "GDOT", linestyle = 'solid', color = "green")
        ax[0,0].set_title(f"ZP=f(DIST) \n {gw_t}t / DISA {disa}°C / {Rating}", fontsize=16, fontweight ="bold") 
        ax[0,0].legend(fontsize=14, loc='lower right')
        ax[0,0].set_xlabel('Distance (NM)', fontsize=14, fontweight ="bold")
        ax[0,0].set_ylabel('ZP (ft)', fontsize=14, fontweight ="bold")

        ax[0,1].plot(Res_Climb_CAS_MACH_df_tmp2.DIST, Res_Climb_CAS_MACH_df_tmp2.CAS, label = "VMO / MMO", linestyle = 'solid', color = "red")
        ax[0,1].plot(Res_Climb_GDOT_df_tmp2.DIST, Res_Climb_GDOT_df_tmp2.CAS, label = "GDOT", linestyle = 'solid', color = "green")
        ax[0,1].set_title(f"CAS=f(DIST) \n {gw_t}t / DISA {disa}°C / {Rating}", fontsize=16, fontweight ="bold") 
        ax[0,1].legend(fontsize=14, loc='upper right')
        ax[0,1].set_xlabel('Distance (NM)', fontsize=14, fontweight ="bold")
        ax[0,1].set_ylabel('CAS (kt)', fontsize=14, fontweight ="bold")

        ax[0,2].plot(Res_Climb_CAS_MACH_df_tmp2.DIST, Res_Climb_CAS_MACH_df_tmp2.MACH, label = "VMO / MMO", linestyle = 'solid', color = "red")        
        ax[0,2].plot(Res_Climb_GDOT_df_tmp2.DIST, Res_Climb_GDOT_df_tmp2.MACH, label = "GDOT", linestyle = 'solid', color = "green")
        ax[0,2].set_title(f"MACH=f(DIST) \n {gw_t}t / DISA {disa}°C / {Rating}", fontsize=16, fontweight ="bold") 
        ax[0,2].legend(fontsize=14, loc='lower center')
        ax[0,2].set_xlabel('Distance (NM)', fontsize=14, fontweight ="bold")
        ax[0,2].set_ylabel('MACH (-)', fontsize=14, fontweight ="bold")
        
        ax[1,0].plot(Res_Climb_CAS_MACH_df_tmp2.TIME, Res_Climb_CAS_MACH_df_tmp2.ZP, label = "VMO / MMO", linestyle = 'solid', color = "red")
        ax[1,0].plot(Res_Climb_GDOT_df_tmp2.TIME, Res_Climb_GDOT_df_tmp2.ZP, label = "GDOT", linestyle = 'solid', color = "green")
        ax[1,0].set_title(f"ZP=f(TIME) \n {gw_t}t / DISA {disa}°C / {Rating}", fontsize=16, fontweight ="bold") 
        ax[1,0].legend(fontsize=14, loc='lower right')
        ax[1,0].set_xlabel('TIME (MIN)', fontsize=14, fontweight ="bold")
        ax[1,0].set_ylabel('ZP (ft)', fontsize=14, fontweight ="bold")
        
        ax[1,1].plot(Res_Climb_CAS_MACH_df_tmp2.ZP, Res_Climb_CAS_MACH_df_tmp2.VZP, label = "VMO / MMO", linestyle = 'solid', color = "red")
        ax[1,1].plot(Res_Climb_GDOT_df_tmp2.ZP, Res_Climb_GDOT_df_tmp2.VZP, label = "GDOT", linestyle = 'solid', color = "green")
        ax[1,1].set_title(f"VZP=f(ZP) \n {gw_t}t / DISA {disa}°C / {Rating}", fontsize=16, fontweight ="bold") 
        ax[1,1].legend(fontsize=14, loc='upper right')
        ax[1,1].set_xlabel('ZP (ft)', fontsize=14, fontweight ="bold")
        ax[1,1].set_ylabel('VZP (ft/min)', fontsize=14, fontweight ="bold")

        ax[1,2].plot(Res_Climb_CAS_MACH_df_tmp2.ZP, Res_Climb_CAS_MACH_df_tmp2.GAMA_AIR_GEOM, label = "VMO / MMO", linestyle = 'solid', color = "red")
        ax[1,2].plot(Res_Climb_GDOT_df_tmp2.ZP, Res_Climb_GDOT_df_tmp2.GAMA_AIR_GEOM, label = "GDOT", linestyle = 'solid', color = "green")
        ax[1,2].set_title(f"GAMMA AIR GEOM=f(ZP) \n {gw_t}t / DISA {disa}°C / {Rating}", fontsize=16, fontweight ="bold") 
        ax[1,2].legend(fontsize=14, loc='upper right')
        ax[1,2].set_xlabel('ZP (ft)', fontsize=14, fontweight ="bold")
        ax[1,2].set_ylabel('GAMMA (deg)', fontsize=14, fontweight ="bold")
        
        plt.subplots_adjust(wspace=0.2, hspace=0.4)

NameError: name 'plt' is not defined

In [ ]:
%%capture
os.chdir(path_main)

Report_Dir_Name = path_reports + "/" + Name_HTML_Report + ".html"
shutil.move ('Mr_SESAR_study.html', Report_Dir_Name)